In [1]:
import gensim
import nltk as nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

%matplotlib inline

In [7]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package punkt to
[nltk_data]     /home/hsoufan@SUADEO.NET/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/hsoufan@SUADEO.NET/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/hsoufan@SUADEO.NET/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/hsoufan@SUADEO.NET/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [3]:
course_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_processed.csv"
course_df = pd.read_csv(course_url)
course_df.head()

,COURSE_ID,TITLE,DESCRIPTION
0,ML0201EN,robots are coming build iot apps with watson ...,have fun with iot and learn along the way if ...
1,ML0122EN,accelerating deep learning with gpu,training complex deep learning models with lar...
2,GPXX0ZG0EN,consuming restful services using the reactive ...,learn how to use a reactive jax rs client to a...
3,RP0105EN,analyzing big data in r using apache spark,apache spark is a popular cluster computing fr...
4,GPXX0Z2PEN,containerizing packaging and running a sprin...,learn how to containerize package and run a ...


In [4]:
course_df["course_text"] = course_df[['TITLE', 'DESCRIPTION']].agg(' '.join, axis=1)
course_df.head()

,COURSE_ID,TITLE,DESCRIPTION,course_text
0,ML0201EN,robots are coming build iot apps with watson ...,have fun with iot and learn along the way if ...,robots are coming build iot apps with watson ...
1,ML0122EN,accelerating deep learning with gpu,training complex deep learning models with lar...,accelerating deep learning with gpu training c...
2,GPXX0ZG0EN,consuming restful services using the reactive ...,learn how to use a reactive jax rs client to a...,consuming restful services using the reactive ...
3,RP0105EN,analyzing big data in r using apache spark,apache spark is a popular cluster computing fr...,analyzing big data in r using apache spark apa...
4,GPXX0Z2PEN,containerizing packaging and running a sprin...,learn how to containerize package and run a ...,containerizing packaging and running a sprin...


In [9]:
def tokenize_course(course: str, keep_only_nouns=True):
    stop_words = set(stopwords.words('english'))
    word_tokens = word_tokenize(course)
    word_tokens = [w for w in word_tokens if (not w.lower() in stop_words) and (not w.isnumeric())]
    if keep_only_nouns:
        filter_list = ['WDT', 'WP', 'WRB', 'FW', 'IN', 'JJR', 'JJS', 'MD', 'PDT', 'POS', 'PRP', 'RB', 'RBR', 'RBS',
                       'RP']
        tags = nltk.pos_tag(word_tokens)
        word_tokens = [w for w, p in tags if p not in filter_list]
    return word_tokens

In [10]:
tokenized_courses = course_df["course_text"].apply(tokenize_course)
tokenized_courses.head()

0    [robots, coming, build, iot, apps, watson, swi...
1    [accelerating, deep, learning, gpu, training, ...
2    [consuming, restful, services, using, reactive...
3    [analyzing, big, data, r, using, apache, spark...
4    [containerizing, packaging, running, spring, b...
Name: course_text, dtype: object

In [18]:
token_dict = gensim.corpora.Dictionary(tokenized_courses)
bows = [token_dict.doc2bow(doc) for doc in tokenized_courses]

In [23]:
doc_indices = []
doc_ids = []
tokens = []
bow_values = []

for doc_index, doc_bow in enumerate(bows):
    for token_index, token_bow in doc_bow:
        doc_indices.append(doc_index)
        doc_ids.append(course_df.iloc[doc_index]["COURSE_ID"])
        tokens.append(token_dict[token_index])
        bow_values.append(token_bow)

bow_df = pd.DataFrame({"doc_index": doc_indices, "doc_id": doc_ids, "token": tokens, "bow_value": bow_values})
bow_df.head()

,doc_index,doc_id,token,bow_value
0,0,ML0201EN,ai,2
1,0,ML0201EN,apps,2
2,0,ML0201EN,build,2
3,0,ML0201EN,cloud,1
4,0,ML0201EN,coming,1
